In [1]:
import pandas as pd

ri_structure = pd.read_parquet("data/processed/ri_structure_dataset_clean.parquet")
print(ri_structure.shape)
ri_structure[["jid", "reduced_formula", "refractive_index"]].head()

(32341, 142)


,jid,reduced_formula,refractive_index
0,JVASP-90856,TiCuSiAs,8.296590
1,JVASP-86097,DyB6,11.873256
2,JVASP-64906,Be2OsRu,14.027763
3,JVASP-64664,Ba4NaBi,10.828656
4,JVASP-86726,LuNi4Sn,15.273380


In [3]:
import pandas as pd
import numpy as np
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms

# Load your dataset
ri_structure = pd.read_parquet("data/processed/ri_structure_dataset_clean.parquet")

# Load fresh JARVIS structures
dft3d = data("dft_3d")
structure_lookup = {x["jid"]: x["atoms"] for x in dft3d}

def clean_atoms_dict(ad):
    return {
        "lattice_mat": np.asarray(ad["lattice_mat"], dtype=float).tolist(),
        "coords": np.asarray(ad["coords"], dtype=float).tolist(),
        "elements": [str(x) for x in np.asarray(ad["elements"]).tolist()],
        "abc": np.asarray(ad["abc"], dtype=float).tolist(),
        "angles": np.asarray(ad["angles"], dtype=float).tolist(),
        "cartesian": bool(np.asarray(ad["cartesian"]).item()),
        "props": [str(x) for x in np.asarray(ad["props"]).tolist()],
    }

# Reattach fresh structure dicts
ri_structure["atoms_raw"] = ri_structure["jid"].map(structure_lookup)

# Test one row
raw_atoms = ri_structure.iloc[0]["atoms_raw"]
clean_atoms = clean_atoms_dict(raw_atoms)
sample_atoms = Atoms.from_dict(clean_atoms)

print(sample_atoms)
print(sample_atoms.num_atoms)
print(sample_atoms.composition)

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
System
1.0
3.566933224304235 0.0 -0.0
0.0 3.566933224304235 -0.0
-0.0 -0.0 9.397075454186664
Ti Cu Si As 
2 2 2 2 
direct
0.7500000000000001 0.7500000000000001 0.784935507022239 Ti
0.25 0.25 0.2150644929777609 Ti
0.25 0.7500000000000001 0.5 Cu
0.7500000000000001 0.25 0.5 Cu
0.25 0.7500000000000001 0.0 Si
0.7500000000000001 0.25 0.0 Si
0.7500000000000001 0.7500000000000001 0.3074869598412097 As
0.25 0.25 0.6925130401587904 As

8
OrderedDict({'Ti': 2, 'Cu': 2, 'Si': 2, 'As': 2})


In [4]:
ri_structure["atoms"] = ri_structure["atoms_raw"].apply(clean_atoms_dict)
ri_structure = ri_structure.drop(columns=["atoms_raw"])

print(ri_structure.shape)
print(ri_structure["atoms"].notna().mean())

(32341, 142)
1.0


In [5]:
ri_structure.to_parquet("data/processed/ri_structure_dataset_cgcnn.parquet", index=False)

In [6]:
import pandas as pd
import numpy as np
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms

ri_structure = pd.read_parquet("data/processed/ri_structure_dataset_clean.parquet")
print(ri_structure.shape)

dft3d = data("dft_3d")
structure_lookup = {x["jid"]: x["atoms"] for x in dft3d}

print(len(structure_lookup))

(32341, 142)
Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
93902


In [7]:
def clean_atoms_dict(ad):
    def to_list(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if hasattr(x, "tolist"):
            return x.tolist()
        return x

    return {
        "lattice_mat": np.array(to_list(ad["lattice_mat"]), dtype=float).tolist(),
        "coords": np.array(to_list(ad["coords"]), dtype=float).tolist(),
        "elements": [str(x) for x in to_list(ad["elements"])],
        "abc": np.array(to_list(ad["abc"]), dtype=float).tolist(),
        "angles": np.array(to_list(ad["angles"]), dtype=float).tolist(),
        "cartesian": bool(np.array(to_list(ad["cartesian"])).reshape(-1)[0]),
        "props": [str(x) for x in to_list(ad["props"])],
    }

ri_structure["atoms_raw"] = ri_structure["jid"].map(structure_lookup)

sample_raw = ri_structure.iloc[0]["atoms_raw"]
sample_atoms = Atoms.from_dict(clean_atoms_dict(sample_raw))

print(sample_atoms)
print(sample_atoms.num_atoms)
print(sample_atoms.composition)

System
1.0
3.566933224304235 0.0 -0.0
0.0 3.566933224304235 -0.0
-0.0 -0.0 9.397075454186664
Ti Cu Si As 
2 2 2 2 
direct
0.7500000000000001 0.7500000000000001 0.784935507022239 Ti
0.25 0.25 0.2150644929777609 Ti
0.25 0.7500000000000001 0.5 Cu
0.7500000000000001 0.25 0.5 Cu
0.25 0.7500000000000001 0.0 Si
0.7500000000000001 0.25 0.0 Si
0.7500000000000001 0.7500000000000001 0.3074869598412097 As
0.25 0.25 0.6925130401587904 As

8
OrderedDict({'Ti': 2, 'Cu': 2, 'Si': 2, 'As': 2})


In [8]:
ri_structure["atoms"] = ri_structure["atoms_raw"].apply(clean_atoms_dict)

print(ri_structure["atoms"].notna().mean())
print(ri_structure[["jid", "reduced_formula", "refractive_index"]].head())

1.0
           jid reduced_formula  refractive_index
0  JVASP-90856        TiCuSiAs          8.296590
1  JVASP-86097            DyB6         11.873256
2  JVASP-64906         Be2OsRu         14.027763
3  JVASP-64664         Ba4NaBi         10.828656
4  JVASP-86726         LuNi4Sn         15.273380


In [9]:
ri_cgcnn_ready = ri_structure[["jid", "reduced_formula", "refractive_index", "atoms"]].copy()

ri_cgcnn_ready.to_parquet(
    "data/processed/ri_cgcnn_ready.parquet",
    index=False
)

print(ri_cgcnn_ready.shape)

(32341, 4)


In [10]:
import pandas as pd
import numpy as np
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import data

In [11]:
ri_cgcnn_ready = pd.read_parquet("data/processed/ri_cgcnn_ready.parquet")
print(ri_cgcnn_ready.shape)
ri_cgcnn_ready[["jid", "reduced_formula", "refractive_index"]].head()

(32341, 4)


,jid,reduced_formula,refractive_index
0,JVASP-90856,TiCuSiAs,8.296590
1,JVASP-86097,DyB6,11.873256
2,JVASP-64906,Be2OsRu,14.027763
3,JVASP-64664,Ba4NaBi,10.828656
4,JVASP-86726,LuNi4Sn,15.273380


In [12]:
dft3d = data("dft_3d")
structure_lookup = {x["jid"]: x["atoms"] for x in dft3d}

def clean_atoms_dict(ad):
    def to_list(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if hasattr(x, "tolist"):
            return x.tolist()
        return x

    return {
        "lattice_mat": np.array(to_list(ad["lattice_mat"]), dtype=float).tolist(),
        "coords": np.array(to_list(ad["coords"]), dtype=float).tolist(),
        "elements": [str(x) for x in to_list(ad["elements"])],
        "abc": np.array(to_list(ad["abc"]), dtype=float).tolist(),
        "angles": np.array(to_list(ad["angles"]), dtype=float).tolist(),
        "cartesian": bool(np.array(to_list(ad["cartesian"])).reshape(-1)[0]),
        "props": [str(x) for x in to_list(ad["props"])],
    }

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.


In [13]:
ri_cgcnn_ready["atoms_raw"] = ri_cgcnn_ready["jid"].map(structure_lookup)

sample_raw = ri_cgcnn_ready.iloc[0]["atoms_raw"]
sample_atoms = Atoms.from_dict(clean_atoms_dict(sample_raw))

print(sample_atoms)
print(sample_atoms.num_atoms)
print(sample_atoms.composition)

System
1.0
3.566933224304235 0.0 -0.0
0.0 3.566933224304235 -0.0
-0.0 -0.0 9.397075454186664
Ti Cu Si As 
2 2 2 2 
direct
0.7500000000000001 0.7500000000000001 0.784935507022239 Ti
0.25 0.25 0.2150644929777609 Ti
0.25 0.7500000000000001 0.5 Cu
0.7500000000000001 0.25 0.5 Cu
0.25 0.7500000000000001 0.0 Si
0.7500000000000001 0.25 0.0 Si
0.7500000000000001 0.7500000000000001 0.3074869598412097 As
0.25 0.25 0.6925130401587904 As

8
OrderedDict({'Ti': 2, 'Cu': 2, 'Si': 2, 'As': 2})


In [14]:
ri_cgcnn_ready["atoms_raw"] = ri_cgcnn_ready["jid"].map(structure_lookup)

sample_raw = ri_cgcnn_ready.iloc[0]["atoms_raw"]
sample_atoms = Atoms.from_dict(clean_atoms_dict(sample_raw))

print(sample_atoms)
print(sample_atoms.num_atoms)
print(sample_atoms.composition)

System
1.0
3.566933224304235 0.0 -0.0
0.0 3.566933224304235 -0.0
-0.0 -0.0 9.397075454186664
Ti Cu Si As 
2 2 2 2 
direct
0.7500000000000001 0.7500000000000001 0.784935507022239 Ti
0.25 0.25 0.2150644929777609 Ti
0.25 0.7500000000000001 0.5 Cu
0.7500000000000001 0.25 0.5 Cu
0.25 0.7500000000000001 0.0 Si
0.7500000000000001 0.25 0.0 Si
0.7500000000000001 0.7500000000000001 0.3074869598412097 As
0.25 0.25 0.6925130401587904 As

8
OrderedDict({'Ti': 2, 'Cu': 2, 'Si': 2, 'As': 2})


In [15]:
ri_cgcnn_ready["atoms"] = ri_cgcnn_ready["atoms_raw"].apply(clean_atoms_dict)
ri_cgcnn_ready = ri_cgcnn_ready.drop(columns=["atoms_raw"])

print(ri_cgcnn_ready["atoms"].notna().mean())

1.0


In [16]:
ri_cgcnn_ready = ri_cgcnn_ready[["jid", "reduced_formula", "refractive_index", "atoms"]].copy()
ri_cgcnn_ready.to_parquet("data/processed/ri_cgcnn_ready.parquet", index=False)
print(ri_cgcnn_ready.shape)

(32341, 4)


In [17]:
import pandas as pd
import numpy as np
from jarvis.core.atoms import Atoms

ri_cgcnn_ready = pd.read_parquet("data/processed/ri_cgcnn_ready.parquet")
print(ri_cgcnn_ready.shape)

def clean_atoms_dict(ad):
    def to_list(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if hasattr(x, "tolist"):
            return x.tolist()
        return x

    return {
        "lattice_mat": np.array(to_list(ad["lattice_mat"]), dtype=float).tolist(),
        "coords": np.array(to_list(ad["coords"]), dtype=float).tolist(),
        "elements": [str(x) for x in to_list(ad["elements"])],
        "abc": np.array(to_list(ad["abc"]), dtype=float).tolist(),
        "angles": np.array(to_list(ad["angles"]), dtype=float).tolist(),
        "cartesian": bool(np.array(to_list(ad["cartesian"])).reshape(-1)[0]),
        "props": [str(x) for x in to_list(ad["props"])],
    }

(32341, 4)


In [18]:
# Validate one structure
sample_raw = ri_cgcnn_ready.iloc[0]["atoms"]
sample_atoms = Atoms.from_dict(clean_atoms_dict(sample_raw))

print(sample_atoms)
print("num_atoms:", sample_atoms.num_atoms)
print("composition:", sample_atoms.composition)

System
1.0
3.566933224304235 0.0 -0.0
0.0 3.566933224304235 -0.0
-0.0 -0.0 9.397075454186664
Ti Cu Si As 
2 2 2 2 
direct
0.7500000000000001 0.7500000000000001 0.784935507022239 Ti
0.25 0.25 0.2150644929777609 Ti
0.25 0.7500000000000001 0.5 Cu
0.7500000000000001 0.25 0.5 Cu
0.25 0.7500000000000001 0.0 Si
0.7500000000000001 0.25 0.0 Si
0.7500000000000001 0.7500000000000001 0.3074869598412097 As
0.25 0.25 0.6925130401587904 As

num_atoms: 8
composition: OrderedDict({'Ti': 2, 'Cu': 2, 'Si': 2, 'As': 2})


In [19]:
# Convert all structures to a cleaned format and verify none fail
cleaned_atoms = []
failures = 0

for ad in ri_cgcnn_ready["atoms"]:
    try:
        cleaned_atoms.append(clean_atoms_dict(ad))
    except Exception:
        cleaned_atoms.append(None)
        failures += 1

ri_cgcnn_ready["atoms_clean"] = cleaned_atoms

print("failures:", failures)
print("coverage:", ri_cgcnn_ready["atoms_clean"].notna().mean())

failures: 0
coverage: 1.0


In [20]:
# Save the cleaned CGCNN-ready table
ri_cgcnn_ready = ri_cgcnn_ready[["jid", "reduced_formula", "refractive_index", "atoms_clean"]].copy()
ri_cgcnn_ready.to_parquet("data/processed/ri_cgcnn_ready_clean.parquet", index=False)

print(ri_cgcnn_ready.shape)

(32341, 4)


In [21]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ri_cgcnn_ready = pd.read_parquet("data/processed/ri_cgcnn_ready_clean.parquet")
print(ri_cgcnn_ready.shape)

# Split by reduced formula so the same chemistry does not leak across splits
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(
    gss.split(ri_cgcnn_ready, groups=ri_cgcnn_ready["reduced_formula"])
)

trainval_df = ri_cgcnn_ready.iloc[trainval_idx].copy()
test_df = ri_cgcnn_ready.iloc[test_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1764706, random_state=42)
train_idx, val_idx = next(
    gss2.split(trainval_df, groups=trainval_df["reduced_formula"])
)

train_df = trainval_df.iloc[train_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

(32341, 4)
train: (22630, 4)
val: (4858, 4)
test: (4853, 4)


In [22]:
train_forms = set(train_df["reduced_formula"])
val_forms = set(val_df["reduced_formula"])
test_forms = set(test_df["reduced_formula"])

print("train ∩ val:", len(train_forms & val_forms))
print("train ∩ test:", len(train_forms & test_forms))
print("val ∩ test:", len(val_forms & test_forms))

train ∩ val: 0
train ∩ test: 0
val ∩ test: 0


In [23]:
train_df[["jid", "reduced_formula", "refractive_index"]].to_parquet(
    "data/processed/cgcnn_train.parquet", index=False
)
val_df[["jid", "reduced_formula", "refractive_index"]].to_parquet(
    "data/processed/cgcnn_val.parquet", index=False
)
test_df[["jid", "reduced_formula", "refractive_index"]].to_parquet(
    "data/processed/cgcnn_test.parquet", index=False
)

In [24]:
print(train_df.columns)

train_df.iloc[0]

Index(['jid', 'reduced_formula', 'refractive_index', 'atoms_clean'], dtype='object')


jid                                                       JVASP-90856
reduced_formula                                              TiCuSiAs
refractive_index                                              8.29659
atoms_clean         {'lattice_mat': [[3.566933224304235, 0.0, -0.0...
Name: 0, dtype: object

In [25]:
sample = train_df.iloc[0]

print(sample["jid"])
print(sample["reduced_formula"])
print(sample["refractive_index"])

JVASP-90856
TiCuSiAs
8.296589660818475


In [26]:
sample["atoms_clean"]

{'lattice_mat': array([array([ 3.56693322,  0.        , -0.        ]),
        array([ 0.        ,  3.56693322, -0.        ]),
        array([-0.        , -0.        ,  9.39707545])], dtype=object),
 'coords': array([array([2.6751975 , 2.6751975 , 7.37610175]),
        array([0.8917325 , 0.8917325 , 2.02097825]),
        array([0.8917325, 2.6751975, 4.69854  ]),
        array([2.6751975, 0.8917325, 4.69854  ]),
        array([0.8917325, 2.6751975, 0.       ]),
        array([2.6751975, 0.8917325, 0.       ]),
        array([2.6751975 , 2.6751975 , 2.88947956]),
        array([0.8917325 , 0.8917325 , 6.50760044])], dtype=object),
 'elements': array(['Ti', 'Ti', 'Cu', 'Cu', 'Si', 'Si', 'As', 'As'], dtype=object),
 'abc': array([3.56693, 3.56693, 9.39708]),
 'angles': array([90., 90., 90.]),
 'cartesian': True,
 'props': array(['', '', '', '', '', '', '', ''], dtype=object)}

In [27]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from pymatgen.core.periodic_table import Element

train_df = pd.read_parquet("data/processed/cgcnn_train.parquet")
val_df = pd.read_parquet("data/processed/cgcnn_val.parquet")
test_df = pd.read_parquet("data/processed/cgcnn_test.parquet")

print(train_df.shape, val_df.shape, test_df.shape)

ModuleNotFoundError: No module named 'torch'